# Preloaded Map-Style Dataset

**Difficulty**: 🟢 Easy

---

### Problem Statement

The most common pattern in PyTorch data loading is a **map-style dataset**: all data lives in memory (or is indexed on disk), and `__getitem__(i)` returns the i-th sample. Combined with `DataLoader`, this gives you batching, shuffling, and multi-worker prefetching for free.

Your task is to implement a map-style `Dataset` that loads a synthetic dataset **entirely into memory** as tensors, and verify that `DataLoader` correctly batches and shuffles it.

---

### Requirements

1. **`PreloadedDataset(X, y)`** — A `torch.utils.data.Dataset` subclass:
   - Stores `X` and `y` as tensors in `__init__`.
   - `__len__` returns the number of samples.
   - `__getitem__(idx)` returns `(X[idx], y[idx])`.

2. **DataLoader integration**:
   - Wrap the dataset in a `DataLoader` with `batch_size=16` and `shuffle=True`.
   - Iterate one full epoch and verify every sample is seen exactly once.

3. **Multi-worker loading**:
   - Use `num_workers=2` and confirm the output is identical in content (though order may differ when shuffled).

---

### Constraints

- ✅ Data must be stored as `torch.Tensor`, not numpy arrays or lists.
- ✅ `__getitem__` must return a tuple `(x_i, y_i)`, not a dict.
- ✅ Must work with `DataLoader`'s batching (automatic collation).

---

<details>
  <summary>💡 Hint</summary>

  - Convert data to tensors in `__init__` with `torch.tensor()` or `torch.as_tensor()`.
  - `DataLoader` calls `__getitem__` for each index in the batch, then stacks the results automatically.
  - To verify all samples are seen, collect all indices or values across batches and check the count.

</details>

---

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Synthetic data: 1000 samples, 10 features, regression target
torch.manual_seed(42)
N, D = 1000, 10
X_data = torch.randn(N, D)
y_data = X_data @ torch.randn(D) + 0.1 * torch.randn(N)

print(f"X shape: {X_data.shape}, y shape: {y_data.shape}")

In [ ]:
class PreloadedDataset(Dataset):
    """Map-style dataset that holds all data in memory as tensors."""

    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        # TODO: Store X and y as tensors
        ...

    def __len__(self):
        # TODO: Return the number of samples
        ...

    def __getitem__(self, idx):
        # TODO: Return the (X[idx], y[idx]) pair
        ...

In [ ]:
# === Tests ===

# Test 1: Basic dataset properties
ds = PreloadedDataset(X_data, y_data)
assert len(ds) == N, f"Expected length {N}, got {len(ds)}"
x_i, y_i = ds[0]
assert x_i.shape == (D,), f"Expected feature shape ({D},), got {x_i.shape}"
assert y_i.shape == (), f"Expected scalar target, got shape {y_i.shape}"
print("PASSED: Dataset has correct length and item shapes")

# Test 2: DataLoader batching
loader = DataLoader(ds, batch_size=16, shuffle=False)
batch_x, batch_y = next(iter(loader))
assert batch_x.shape == (16, D), f"Batch X shape wrong: {batch_x.shape}"
assert batch_y.shape == (16,), f"Batch y shape wrong: {batch_y.shape}"
print("PASSED: DataLoader produces correct batch shapes")

# Test 3: Full epoch — every sample seen exactly once
loader_shuffled = DataLoader(ds, batch_size=16, shuffle=True)
all_y = []
for bx, by in loader_shuffled:
    all_y.append(by)
all_y = torch.cat(all_y)
assert len(all_y) == N, f"Expected {N} samples in one epoch, got {len(all_y)}"
assert torch.allclose(all_y.sort().values, y_data.sort().values), \
    "Not all samples were seen — values don't match after sorting"
print("PASSED: Full epoch sees every sample exactly once")

# Test 4: Multi-worker loading
loader_workers = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
all_y_w = torch.cat([by for _, by in loader_workers])
assert torch.allclose(all_y_w.sort().values, y_data.sort().values), \
    "Multi-worker loading lost or duplicated samples"
print("PASSED: num_workers=2 produces correct results")

print("\nAll tests passed!")